## **Exam Project Notebook (Preprocessing) - Foundations of Deep Learning**  
### *Classification of Aquatic Animals Images Using Convolution Neural Networks*  

---

**Authors:** *Daniela Lorriz Bautista (ID 900074), Matteo Fumagalli (ID 894468)*  
**University:** *University of Milano-Bicocca*  
**Degree Program:** *Master’s Degree in Data Science*   
**Academic Year:** *2025/2026*

---

### **Notebook Description**
This notebook implements the preprocessing pipeline designed to reconstruct images from Google's Quick, Draw! dataset. In particular, it targets a custom 10-class subsample of aquatic animals in order to processes the raw `.ndjson` files containing vector strokes coordinates and transforms them into standard image formats ready for training.


---

## **0. Environment setup**

In [ ]:
# Install external libraries
!pip install requests

In [ ]:
# Import libraries
import cv2
import json
import numpy as np
import os
import random
import shutil
import requests
import zipfile

# Set seed for reproducibility
SEED = 3
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)

---

## **1. Drive Connection and Data Loading**

In [ ]:
# Connect to drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Define the classes
classes = ['crab', 'crocodile', 'dolphin', 'fish', 'frog',
           'lobster', 'octopus', 'sea turtle', 'shark', 'whale']

# Create raw folder
os.makedirs('/content/raw', exist_ok=True)

# Download raw files from Google Cloud Storage
print('-> Downloading raw files from Google Cloud Storage')
for class_name in classes:
    # Replace spaces with %20 (for the class sea turtle)
    url_class_name = class_name.replace(' ', '%20')

    # Define the url and the output path
    url = ('https://storage.googleapis.com/quickdraw_dataset/'
           f'full/raw/{url_class_name}.ndjson')
    output_path = f'/content/raw/full_raw_{class_name}.ndjson'

    # Check if the file has already been downloaded
    if not os.path.exists(output_path):
        print(f'\nClass: {class_name}')

        try:
          # Define a HTTP GET request and save the result
          with requests.get(url, stream=True, timeout=30) as response:
            response.raise_for_status()
            with open(output_path, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    file.write(chunk)
            print(' -> completed')
        except requests.RequestException as e:
            print(f' -> download error: {e}')

-> Downloading raw files from Google Cloud Storage

Class: crab
 -> completed

Class: crocodile
 -> completed

Class: dolphin
 -> completed

Class: fish
 -> completed

Class: frog
 -> completed

Class: lobster
 -> completed

Class: octopus
 -> completed

Class: sea turtle
 -> completed

Class: shark
 -> completed

Class: whale
 -> completed


---

## **2. Image Reconstruction and Zip Folder Creation**

In [ ]:
def drawing_to_image(drawing,
                     image_size=64, canvas_size=256,
                     padding=16, stroke_width=5):
    '''Convert a JSON drawing into a 64x64 image'''
    # Create a blank grayscale canvas
    image = np.zeros((canvas_size, canvas_size), dtype=np.uint8)

    # Extract x and y coordinates from all strokes
    all_x = [x for stroke in drawing for x in stroke[0]]
    all_y = [y for stroke in drawing for y in stroke[1]]

    # Compute the bounding box and the dimension of the image
    min_x, max_x = min(all_x), max(all_x)
    min_y, max_y = min(all_y), max(all_y)
    width = max_x - min_x
    height = max_y - min_y

    # Define the scale factor to fit drawing into canvas
    # < 1: scale down / > 1: scale up
    scale = (canvas_size - 2 * padding) / max(width, height)

    # Define the offset to place drawing in the middle of the canvas
    x_offset = (canvas_size - width * scale) / 2
    y_offset = (canvas_size - height * scale) / 2

    # Obtain scaled and centered pixel coordinates
    for stroke in drawing:
        points = []
        for x, y in zip(stroke[0], stroke[1]):
            new_x = int((x - min_x) * scale + x_offset)
            new_y = int((y - min_y) * scale + y_offset)
            points.append((new_x, new_y))

        # Draw white lines between consecutive points in the stroke
        for i in range(len(points) - 1):
            cv2.line(image, points[i], points[i+1], 255,
                     stroke_width, cv2.LINE_AA)

    # Resize final image to 64x64 by downsampling
    return cv2.resize(image, (image_size, image_size),
                      interpolation=cv2.INTER_AREA)

In [ ]:
def analyse_class(ndjson_path):
    '''Analyse a NDJSON file and compute statistics over recognized drawings'''
    total_samples = 0
    recognized_samples = 0
    point_counts = []
    stroke_counts = []

    # Open each NDJSON file composed of one JSON object for each line
    with open(ndjson_path, 'r') as file:
        for line in file:
            total_samples += 1
            sample = json.loads(line)

            # Skip drawings not recognized
            if not sample['recognized']:
                continue

            recognized_samples += 1

            # Extract drawing representation
            drawing = sample['drawing']

            # Compute the number of points and strokes
            num_points = sum(len(stroke[0]) for stroke in drawing)
            num_strokes = len(drawing)
            point_counts.append(num_points)
            stroke_counts.append(num_strokes)

    # Compute and return summary statistics
    point_counts = np.array(point_counts)
    stroke_counts = np.array(stroke_counts)

    return {
        'total_samples': total_samples,
        'recognized_samples': recognized_samples,

        'points': {
            'mean': float(np.mean(point_counts)),
            'std': float(np.std(point_counts)),
            'min': int(np.min(point_counts)),
            'max': int(np.max(point_counts)),
            'p5': float(np.percentile(point_counts, 5)),
            'p95': float(np.percentile(point_counts, 95)),
        },

        'strokes': {
            'mean': float(np.mean(stroke_counts)),
            'std': float(np.std(stroke_counts)),
            'min': int(np.min(stroke_counts)),
            'max': int(np.max(stroke_counts)),
            'p5': float(np.percentile(stroke_counts, 5)),
            'p95': float(np.percentile(stroke_counts, 95)),
        }
    }


def preprocess_file(ndjson_path, class_name,
                    output_directory, max_images=20000):
    '''Preprocess an entire NDJSON file of a class'''

    # Display summary statistics
    stats = analyse_class(ndjson_path)
    p5, p95 = stats['points']['p5'], stats['points']['p95']

    print(f'\n==== CLASS: {class_name} ====')
    print(f"Total samples: {stats['total_samples']}")
    print(f"Recognized samples: {stats['recognized_samples']}")
    print('\nPoint statistics:')
    print(f"Mean: {stats['points']['mean']:.2f}")
    print(f"Std: {stats['points']['std']:.2f}")
    print(f'5th percentile: {p5:.2f}')
    print(f'95th percentile: {p95:.2f}')

    # Create train, validation, and test directories
    train_directory = os.path.join(
        output_directory, 'train', class_name)
    validation_directory = os.path.join(
        output_directory, 'validation', class_name)
    test_directory = os.path.join(
        output_directory, 'test', class_name)

    os.makedirs(train_directory, exist_ok=True)
    os.makedirs(validation_directory, exist_ok=True)
    os.makedirs(test_directory, exist_ok=True)

    # Open and shuffle the entire file
    with open(ndjson_path, 'r') as file:
        all_lines = file.readlines()
    random.shuffle(all_lines)

    # Extract valid drawings
    valid_drawings = []
    for line in all_lines:
        sample = json.loads(line)

        # Skip drawings not recognized
        if not sample['recognized']:
            continue

        # Extract drawing representation
        drawing = sample['drawing']
        num_points = sum(len(stroke[0]) for stroke in drawing)

        # Skip drawings with a number of points lower than the
        # 5-th percentile or greater than the 95-th percentile
        if num_points < p5 or num_points > p95:
            continue

        # Extract at most 20,000 images per class
        valid_drawings.append(drawing)
        if len(valid_drawings) >= max_images:
            break

    # Extract train (70%), validation (15%), and test (15%) sets
    num_valid = len(valid_drawings)
    train_end = int(0.70 * num_valid)
    validation_end = int(0.85 * num_valid)
    train_drawings = valid_drawings[:train_end]
    validation_drawings = valid_drawings[train_end:validation_end]
    test_drawings = valid_drawings[validation_end:]

    def process_and_save(drawings, folder, prefix):
        '''Save valid images from a list of drawings into a folder'''
        count = 0
        for i, drawing in enumerate(drawings):
            image = drawing_to_image(drawing)

            if image is None:
                continue

            cv2.imwrite(
                os.path.join(folder, f'{class_name}_{prefix}_{i}.png'), image)
            count += 1

        return count

    # Save train, validation, and test drawings
    train_samples = process_and_save(train_drawings, train_directory, 'train')
    validation_samples = process_and_save(
        validation_drawings, validation_directory, 'validation')
    test_samples = process_and_save(test_drawings, test_directory, 'test')

    print('\nTotal images saved: '
          f'{train_samples + validation_samples + test_samples}')
    print(f'Train images: {train_samples}')
    print(f'Validation images: {validation_samples}')
    print(f'Test images: {test_samples}')

In [ ]:
# Preprocess all classes
for class_name in classes:
    ndjson_path = f'/content/raw/full_raw_{class_name}.ndjson'
    preprocess_file(ndjson_path, class_name, '/content/processed')
print('\n-> Preprocessing completed!')

# Create the final zip folder into Google Drive
final_zip_path = (
    '/content/drive/MyDrive/foundations-of-deep-learning/data/quickdraw_dataset')
shutil.make_archive(final_zip_path, 'zip', root_dir='/content/processed')

# Remove temporary folders
shutil.rmtree('/content/processed')
shutil.rmtree('/content/raw')

print('\n-> Final zip folder created!')


==== CLASS: crab ====
Total samples: 126930
Recognized samples: 106466

Point statistics:
Mean: 268.78
Std: 103.92
5th percentile: 138.00
95th percentile: 466.00

Total images saved: 20000
Train images: 14000
Validation images: 3000
Test images: 3000

==== CLASS: crocodile ====
Total samples: 127932
Recognized samples: 106410

Point statistics:
Mean: 280.96
Std: 113.89
5th percentile: 132.00
95th percentile: 492.00

Total images saved: 20000
Train images: 14000
Validation images: 3000
Test images: 3000

==== CLASS: dolphin ====
Total samples: 121613
Recognized samples: 110335

Point statistics:
Mean: 210.95
Std: 83.26
5th percentile: 104.00
95th percentile: 367.00

Total images saved: 20000
Train images: 14000
Validation images: 3000
Test images: 3000

==== CLASS: fish ====
Total samples: 134150
Recognized samples: 126420

Point statistics:
Mean: 144.61
Std: 65.34
5th percentile: 68.00
95th percentile: 268.00

Total images saved: 20000
Train images: 14000
Validation images: 3000
Test 